In [24]:
import pygame
import numpy as np
import random

# 游戏环境
class SnakeGame:
    def __init__(self, width=10, height=10):
        self.width = width
        self.height = height
        self.reset()

    def reset(self):
        self.snake = [(self.width // 2, self.height // 2)]
        self.direction = (0, 0)
        self.food = self._place_food()
        self.score = 0
        self.game_over = False
        return self._get_state()

    def _place_food(self):
        while True:
            food = (random.randint(0, self.width - 1), random.randint(0, self.height - 1))
            if food not in self.snake:
                return food

    def _get_state(self):
        state = np.zeros((self.width, self.height), dtype=np.float32)
        for x, y in self.snake:
            state[x, y] = 1.0
        state[self.food[0], self.food[1]] = 0.5
        return state

def step(self, action):
    if self.game_over:
        return self._get_state(), self.score, self.game_over

    # 更新方向
    if action == 0:  # 上
        new_dir = (-1, 0)
    elif action == 1:  # 下
        new_dir = (1, 0)
    elif action == 2:  # 左
        new_dir = (0, -1)
    elif action == 3:  # 右
        new_dir = (0, 1)
    else:
        new_dir = self.direction

    # 检查方向是否相反
    if (new_dir[0] == -self.direction[0] and new_dir[1] == -self.direction[1]):
        new_dir = self.direction

    self.direction = new_dir

    # 移动蛇
    new_head = (self.snake[0][0] + self.direction[0], self.snake[0][1] + self.direction[1])

    # 检查碰撞
    if (new_head[0] < 0 or new_head[0] >= self.width or
        new_head[1] < 0 or new_head[1] >= self.height or
        new_head in self.snake):
        self.game_over = True
        return self._get_state(), -1, self.game_over  # 碰撞惩罚

    self.snake.insert(0, new_head)

    # 检查是否吃到食物
    if new_head == self.food:
        self.score += 1
        self.food = self._place_food()
        reward = 1  # 吃到食物的奖励
    else:
        self.snake.pop()
        reward = 0.01  # 每存活一步的奖励

    return self._get_state(), reward, self.game_over

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# 神经网络模型
class QNetwork(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(QNetwork, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.fc3 = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)
        return x

# Q-learning 算法
class QLearning:
    def __init__(self, input_size, hidden_size, output_size, lr=0.001, gamma=0.99):
        self.q_network = QNetwork(input_size, hidden_size, output_size)
        self.optimizer = optim.Adam(self.q_network.parameters(), lr=lr)
        self.gamma = gamma

    def get_action(self, state, epsilon=0.1):
        if random.random() < epsilon:
            return random.randint(0, 3)
        else:
            state = torch.FloatTensor(state.flatten())
            q_values = self.q_network(state)
            return torch.argmax(q_values).item()

    def train(self, state, action, reward, next_state, done):
        state = torch.FloatTensor(state.flatten())
        next_state = torch.FloatTensor(next_state.flatten())
        q_values = self.q_network(state)
        next_q_values = self.q_network(next_state)

        target = reward + (1 - done) * self.gamma * torch.max(next_q_values)
        loss = nn.MSELoss()(q_values[action], target)

        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

In [27]:
# 训练参数
num_episodes = 1000
epsilon = 1.0
epsilon_decay = 0.995
min_epsilon = 0.01

# 初始化环境和Q-learning
env = SnakeGame()
input_size = env.width * env.height
output_size = 4
q_learning = QLearning(input_size, 128, output_size)

# 训练循环
for episode in range(num_episodes):
    state = env.reset()
    total_reward = 0
    done = False

    while not done:
        action = q_learning.get_action(state, epsilon)
        next_state, reward, done = env.step(action)
        q_learning.train(state, action, reward, next_state, done)
        state = next_state
        total_reward += reward

    epsilon = max(min_epsilon, epsilon * epsilon_decay)
    print(f"Episode: {episode}, Total Reward: {total_reward}, Epsilon: {epsilon}")

AttributeError: 'SnakeGame' object has no attribute 'step'

In [23]:
def render(env):
    pygame.init()
    cell_size = 40
    screen = pygame.display.set_mode((env.width * cell_size, env.height * cell_size))
    clock = pygame.time.Clock()

    while True:
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                pygame.quit()
                return

        screen.fill((0, 0, 0))
        for x, y in env.snake:
            pygame.draw.rect(screen, (0, 255, 0), (y * cell_size, x * cell_size, cell_size, cell_size))
        pygame.draw.rect(screen, (255, 0, 0), (env.food[1] * cell_size, env.food[0] * cell_size, cell_size, cell_size))
        pygame.display.flip()
        clock.tick(10)

# 在训练结束后可视化游戏
render(env)